# DengueCast v4 — Exploração dos Dados

**Dataset:** [`pedrolucassantanaf/dengue-tcc2-data`](https://huggingface.co/datasets/pedrolucassantanaf/dengue-tcc2-data)  
**Dashboard:** [modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app](https://modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app)  
**Pipeline:** v4 — 137 mesorregiões, 2001–2026, 266 features (SINAN + INMET)

---

Este notebook explora os dados do pipeline v4 do DengueCast: volumetria, distribuição do alvo, cobertura espacial e temporal, correlações de features e verificação de vazamento de dados.

In [ ]:
%%capture
!pip install huggingface_hub plotly kaleido

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download

IS_KAGGLE = "KAGGLE_DATA_DIR" in os.environ

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

REPO = os.environ.get("HF_REPO_ID", "pedrolucassantanaf/dengue-tcc2-data")

def hf_download(path):
    return hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)

print(f"Ambiente: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Repo: {REPO}")

## 1. Carregando os dados

Usamos a **Variante B** (série completa 2001–2022 no treino) como referência principal.

In [ ]:
train = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/train.parquet"))
val = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/val.parquet"))
test = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/test.parquet"))

schema = pd.read_csv(hf_download("data/model_ready_v4/feature_schema_v4.csv"))

df = pd.concat([train, val, test], ignore_index=True)
print(f"Total de linhas: {len(df):,}")
print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Volumetria e cobertura

In [ ]:
n_meso = df["cod_mesorregiao"].nunique()
n_ufs = df["cod_mesorregiao"].astype(str).str[:2].nunique()
anos = sorted(df["ano"].unique())

print(f"Mesorregiões: {n_meso}")
print(f"UFs: {n_ufs}")
print(f"Cobertura temporal: {anos[0]}–{anos[-1]} ({len(anos)} anos)")
print(f"Semanas epidemiológicas: {df['semana_epidemiologica'].min()}–{df['semana_epidemiologica'].max()}")
print(f"Features: {len(schema)} (SINAN: {len(schema[schema['category']=='sinan'])}, INMET: {len(schema[schema['is_climate']==True])})")

In [ ]:
vol = df.groupby("ano").size().reset_index(name="linhas")
vol["split"] = vol["ano"].apply(
    lambda y: "treino" if y <= 2022 else ("validação" if y == 2023 else "teste")
)

fig = px.bar(vol, x="ano", y="linhas", color="split",
             color_discrete_map={"treino": "#5046e5", "validação": "#f59e0b", "teste": "#ef4444"},
             title="Linhas por ano — com anotação de split")
fig.update_layout(xaxis_title="Ano", yaxis_title="Linhas",
                  font=dict(family="DM Sans"), template="plotly_white")
fig.show()

## 3. Distribuição do alvo: `notificacoes_t4`

O alvo é o número de notificações de dengue 4 semanas à frente. Usamos `log1p` para treinar porque a distribuição é fortemente assimétrica.

In [ ]:
target = df["notificacoes_t4"].dropna()

print("Estatísticas de notificacoes_t4:")
print(f"  Min: {target.min():.0f}")
print(f"  Mediana: {target.median():.0f}")
print(f"  Média: {target.mean():.1f}")
print(f"  P95: {target.quantile(0.95):.0f}")
print(f"  P99: {target.quantile(0.99):.0f}")
print(f"  Max: {target.max():.0f}")
print(f"  Zeros: {(target == 0).sum()} ({(target == 0).mean()*100:.1f}%)")

fig = go.Figure()
fig.add_trace(go.Histogram(x=np.log1p(target), nbinsx=100, name="log1p(notificacoes_t4)",
                           marker_color="#5046e5"))
fig.update_layout(title="Distribuição de log1p(notificacoes_t4)",
                  xaxis_title="log1p(notificacoes_t4)", yaxis_title="Frequência",
                  font=dict(family="DM Sans"), template="plotly_white")
fig.show()

## 4. Distribuição dos níveis de risco (`risco_surto_t4`)

4 categorias baseadas no canal endêmico:
- **0 — Baixo**: abaixo do canal
- **1 — Moderado**: dentro do canal
- **2 — Alto**: acima do canal
- **3 — Surto**: acima do limiar epidêmico

In [ ]:
risk_labels = {0: "Baixo", 1: "Moderado", 2: "Alto", 3: "Surto"}
risk_colors = {"Baixo": "#22c55e", "Moderado": "#f59e0b", "Alto": "#f97316", "Surto": "#ef4444"}

risk = df["risco_surto_t4"].map(risk_labels).value_counts().reindex(
    ["Baixo", "Moderado", "Alto", "Surto"]
)
risk_pct = risk / risk.sum() * 100

print("Distribuição de risco:")
for label, count in risk.items():
    print(f"  {label}: {count:,} ({risk_pct[label]:.1f}%)")

fig = px.bar(x=risk.index, y=risk.values,
             color=risk.index, color_discrete_map=risk_colors,
             title="Distribuição dos 4 níveis de risco")
fig.update_layout(xaxis_title="Nível de risco", yaxis_title="Contagem",
                  showlegend=False, font=dict(family="DM Sans"), template="plotly_white")
fig.show()

## 5. Série temporal agregada

Total de notificações por ano (soma de todas as mesorregiões), com anotação de treino/validação/teste.

In [ ]:
ts = df.groupby("ano")["notificacoes_t4"].sum().reset_index()
ts["split"] = ts["ano"].apply(
    lambda y: "treino" if y <= 2022 else ("validação" if y == 2023 else "teste")
)

fig = px.line(ts, x="ano", y="notificacoes_t4", color="split",
              color_discrete_map={"treino": "#5046e5", "validação": "#f59e0b", "teste": "#ef4444"},
              markers=True, title="Total de notificações por ano")
fig.update_layout(xaxis_title="Ano", yaxis_title="Total notificações (t+4)",
                  font=dict(family="DM Sans"), template="plotly_white")
fig.show()

## 6. Cobertura INMET

Porcentagem de semanas-mesorregiões com dados climáticos presentes.

In [ ]:
inmet_cols = schema[schema["is_climate"] == True]["feature"].tolist()
inmet_bruto = schema[schema["category"] == "inmet_bruto"]["feature"].tolist()

if inmet_bruto:
    has_inmet = df[inmet_bruto].notna().any(axis=1)
    coverage_total = has_inmet.mean() * 100

    cov_by_year = df.groupby("ano").apply(lambda g: g[inmet_bruto].notna().any(axis=1).mean() * 100)
    cov_df = cov_by_year.reset_index()
    cov_df.columns = ["ano", "cobertura_pct"]

    print(f"Cobertura INMET total: {coverage_total:.1f}%")

    fig = px.bar(cov_df, x="ano", y="cobertura_pct",
                 title="Cobertura INMET por ano (% de semanas-mesorregiões com dados)",
                 color_discrete_sequence=["#5046e5"])
    fig.update_layout(xaxis_title="Ano", yaxis_title="Cobertura (%)",
                      font=dict(family="DM Sans"), template="plotly_white",
                      yaxis=dict(range=[0, 105]))
    fig.show()
else:
    print("Colunas INMET bruto não encontradas no schema.")

## 7. Mapa de calor: correlação top-20 features

Separando features SINAN e INMET pelo `feature_schema_v4.csv`.

In [ ]:
features = schema["feature"].tolist()
target_col = "notificacoes_t4"

corr_with_target = df[features + [target_col]].corr()[target_col].drop(target_col).abs().sort_values(ascending=False)
top20 = corr_with_target.head(20).index.tolist()

corr_matrix = df[top20].corr()

labels = []
for f in top20:
    cat = schema[schema["feature"] == f]["category"].values[0]
    prefix = "[S]" if cat == "sinan" else "[I]"
    labels.append(f"{prefix} {f[:30]}")

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values, x=labels, y=labels,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1
))
fig.update_layout(title="Correlação top-20 features (mais correlacionadas com o alvo)",
                  width=800, height=700, font=dict(family="DM Sans", size=9),
                  template="plotly_white")
fig.show()

## 8. Verificação de vazamento de dados

Confirmamos que colunas-alvo e metadados **NÃO** estão nas features de treino.

In [ ]:
forbidden = [
    "notificacoes_t4", "risco_surto_t4",
    "risco_surto_t4_com_vazamento", "threshold_source",
    "cod_mesorregiao", "ano", "semana_epidemiologica"
]

feature_set = set(schema["feature"].tolist())
all_cols = set(df.columns)

print("Verificação de vazamento:")
leaks_found = False
for col in forbidden:
    in_features = col in feature_set
    in_data = col in all_cols
    status = "VAZAMENTO" if in_features else "OK"
    if in_features:
        leaks_found = True
    print(f"  [{status}] {col} — no dataset: {in_data}, nas features: {in_features}")

if not leaks_found:
    print("\n✓ Nenhum vazamento detectado. Features são seguras para treino.")
else:
    print("\n✗ ATENÇÃO: Vazamento detectado! Remova as colunas acima das features.")

## 9. Resumo

- **137 mesorregiões**, cobrindo todas as 27 UFs do Brasil
- **2001–2026**: 22 anos de treino + 1 de validação + ~2 de teste
- **266 features**: 110 SINAN (epidemiológicas) + 156 INMET (climáticas)
- Alvo altamente assimétrico — treinamos em `log1p(notificacoes_t4)`
- ~10% das observações são classe Surto (desbalanceamento moderado)
- Nenhum vazamento de dados detectado nas features

**Próximo notebook:** [v4_02_treino](https://www.kaggle.com/pedrolucassantanaf/denguecast-v4-02-treino) — treino e avaliação do XGBoost